# Stacking

Les modèles de base sont déjà entraînés : seules leurs prédictions sont nécessaires.
On entraîne une régression linéaire sur les prédictions OOF, puis on l’applique aux submissions test.
Tous les fichiers sont alignés par `id`.

**Ajouter un modèle :** une ligne dans `MODELS` (cellule 2). Le modèle doit avoir écrit `Stacking/oof_<nom>.csv` et sa submission test (voir `folds.ipynb`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from IPython.display import display

# Fonctionne depuis Stacking, le dossier du projet ou la racine du dépôt.
ROOT = next(p.resolve() for p in [Path(".."), Path("."), Path("eth-aml-2026-project-1")]
            if (p / "sample.csv").exists() and (p / "Stacking").is_dir())
STACKING = ROOT / "Stacking"


In [8]:
def read_predictions(path, name):
    frame = pd.read_csv(path, index_col="id")
    assert frame.index.is_unique, f"IDs dupliqués : {path}"
    assert frame.shape[1] == 1, f"Une seule colonne de prédictions attendue : {path}"
    return frame.iloc[:, 0].rename(name)

# Un modèle = une ligne : nom -> sa submission test (format sample.csv). Ses OOF sont dans Stacking/oof_<nom>.csv.
MODELS = {
    "tabpfn": ROOT / "TabPFN" / "submission_tabpfn.csv",
    "gp": ROOT / "Gaussian_process" / "submission_gp.csv",
}

oof = [read_predictions(STACKING / f"oof_{name}.csv", name) for name in MODELS]
assert all(set(o.index) == set(oof[0].index) for o in oof), "Les IDs OOF diffèrent"
X_meta = pd.concat(oof, axis=1).loc[oof[0].index]
y = pd.read_csv(ROOT / "y_train.csv", index_col="id")["y"].loc[X_meta.index]
folds = pd.read_csv(STACKING / "folds.csv", index_col="id")["fold"].loc[X_meta.index]
assert np.isfinite(X_meta.to_numpy()).all()
assert np.isfinite(y.to_numpy()).all()
assert folds.notna().all() and folds.nunique() >= 2
print(f"{len(X_meta)} observations OOF, {folds.nunique()} folds, modèles : {list(MODELS)}")
display(X_meta.head())

1152 observations OOF, 5 folds, modèles : ['tabpfn', 'gp']


,tabpfn,gp
id,,
0.0,73.879739,76.057769
1.0,55.548655,53.784552
2.0,69.056353,64.730145
4.0,72.919756,74.497568
5.0,61.381352,61.671463


## Validation exploratoire

Le méta-modèle est évalué sur les folds partagés : chaque prédiction est faite par une régression entraînée sur les autres folds.
La moyenne simple des modèles est une référence sans entraînement.

**Limite :** les OOF sont déjà calculées. Les modèles de base ayant produit les OOF des autres folds peuvent avoir vu les cibles du fold de validation du méta-modèle. Cette évaluation ne remplace donc pas une validation imbriquée de toute la chaîne. Le prétraitement et le choix des hyperparamètres peuvent également introduire un optimisme s’ils ont utilisé toutes les données.


In [9]:
stack_oof = pd.Series(np.nan, index=X_meta.index, name="stacking")
for fold in sorted(folds.unique()):
    train, valid = folds != fold, folds == fold
    meta = LinearRegression().fit(X_meta.loc[train], y.loc[train])
    stack_oof.loc[valid] = meta.predict(X_meta.loc[valid])

predictions = {name: X_meta[name] for name in MODELS} | {
    "Moyenne simple": X_meta.mean(axis=1),
    "Stacking linéaire (CV exploratoire)": stack_oof,
}
metrics = pd.DataFrame({
    name: {"R2": r2_score(y, pred),
           "RMSE": np.sqrt(mean_squared_error(y, pred)),
           "MAE": mean_absolute_error(y, pred)}
    for name, pred in predictions.items()
}).T
display(metrics.round(5))

,R2,RMSE,MAE
tabpfn,0.77719,4.42085,3.13914
gp,0.68956,5.21831,3.95450
Moyenne simple,0.76424,4.54755,3.33110
Stacking linéaire (CV exploratoire),0.78002,4.39274,3.12149


## Entraînement final et submissions

La régression finale utilise toutes les OOF. Les submissions de base (fichiers de `MODELS`) doivent provenir de modèles entraînés sur toutes les données, avec les mêmes choix de modèle et de prétraitement que pour leurs OOF.
Les coefficients sont libres : ils ne sont pas contraints à être positifs ni à sommer à 1.


In [10]:
meta_model = LinearRegression().fit(X_meta, y)
display(pd.Series(meta_model.coef_, index=X_meta.columns, name="coefficient"))
print(f"Intercept : {meta_model.intercept_:.6f}")

sample = pd.read_csv(ROOT / "sample.csv")
assert sample["id"].is_unique
test = [read_predictions(path, name) for name, path in MODELS.items()]
assert all(set(t.index) == set(sample["id"]) for t in test), "IDs test différents"
X_test_meta = pd.concat(test, axis=1).loc[sample["id"], X_meta.columns]
assert np.isfinite(X_test_meta.to_numpy()).all()

submission = sample.copy()
submission["y"] = meta_model.predict(X_test_meta)
assert np.isfinite(submission["y"]).all()
submission.to_csv(STACKING / "submission_stacking.csv", index=False)
print(f"Enregistré : submission_stacking.csv ({len(submission)} lignes)")
display(submission.head())

tabpfn    0.884914
gp        0.153474
Name: coefficient, dtype: float64

Intercept : -2.871740
Enregistré : submission_stacking.csv (776 lignes)


,id,y
0,0.0,60.887245
1,1.0,76.168266
2,2.0,70.308835
3,3.0,74.558632
4,4.0,73.590873
